# Region of attraction: three Kuramoto oscillators

This notebook computes the set of states from which convergence of three
Kuramoto oscillators to synchrony, at rate 1, is certified. It uses
`verify_roa` with `trim=True`.

Requirements: Python 3.10 or later and
`pip install "pyddrv[jax,examples] @ git+https://github.com/NetDLab/pyDDRV"`.
See `examples/notebooks/README.md`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from pyddrv import verify_roa
from pyddrv.systems.fields_jax import kuramoto_reduced_jax

## The system

With phases `θ_i` and `φ_i = θ_i − θ_n`, the state `φ` has dimension
`d = n − 1 = 2` and synchrony is `φ = 0`. `kuramoto_reduced_jax` returns the
field and the bound `2k(n − 1)/n` on its one-sided Lipschitz constant in the
max-norm. The bound holds for all states, so no estimation is needed and it is
passed directly as `L=`.

In [ ]:
k, n = 10.0, 3
f, L_bound = kuramoto_reduced_jax(k=k, n=n)
R, d = np.pi, n - 1
print("closed-form L bound:", round(L_bound, 4))

## Computing the region

`verify_roa` takes the target rate `alpha` and returns the cubes from which
convergence at that rate is certified. With `trim=True` a second pass also
requires each certifying trajectory to lie inside the region found in the first
pass at the return time.

In [ ]:
import time
t0 = time.time()
roa = verify_roa(f, R=R, d=d, alpha=1.0, L=L_bound, norm="inf",
                 tau=1.9, eps=np.pi/81, max_refine=6,
                 trim=True, raster_n=729, max_seconds=120)
print(roa.summary())
print(f"certified fraction of Q_R: {roa.volume/(2*R)**d:.1%}, "
      f"wall time {time.time()-t0:.1f}s")

About 82% of the box is certified. The two regions left uncovered surround the
splay states, where the phase differences are ±2π/3. These are unstable
equilibria: trajectories that start near them stay close for a long time before
synchronizing, so convergence at rate 1 cannot be certified there.

## Plot

`plot_roa_2d` draws the certified cubes colored by width: large cubes inside the
region, small ones along its boundary. The small square hole at the origin is
the excluded ball around the equilibrium.

In [ ]:
from pyddrv.viz import plot_roa_2d

ax = plot_roa_2d(roa, color_by="width")           # color by cube width
ax.set_xlabel(r"$\phi_1$"); ax.set_ylabel(r"$\phi_2$")
ax.set_title(f"Kuramoto (k={k:g}, n={n}): certified 1-RoA of sync")
plt.show()

Coloring by split depth (`color_by="depth"`) shows how many times each cube was
divided before it was certified.

In [ ]:
ax = plot_roa_2d(roa, color_by="depth")
ax.set_xlabel(r"$\phi_1$"); ax.set_ylabel(r"$\phi_2$")
ax.set_title("Certified 1-RoA, colored by split depth")
plt.show()